# Service Path Decomposition Algorithm

**Task:** P7-SERVICE-PATH-ALGORITHM  
**Author:** @data-engineer  
**Schema:** `cdm_tmforum.provable_resilience_intelligence`

Rebuilds `service_path` with **topologically valid** hop sequences:
- Each consecutive hop shares a site endpoint (hop N site_z = hop N+1 site_a)
- Working paths use shortest BFS path through site graph
- Protect paths use edge-disjoint alternate routes for true SRLG diversity
- Preserves all FK integrity to `physical_resource_span` and `service_inventory`

**Finding:** Prior synthetic data had only 1.1% topological continuity (random span assignment). This algorithm enforces 100% continuity via graph traversal.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType, TimestampType
from collections import defaultdict, deque
import random
import hashlib
import math
from datetime import datetime

CATALOG = "cdm_tmforum"
SCHEMA = "provable_resilience_intelligence"
random.seed(42)

# Load spans and sites
spans_df = spark.table(f"{CATALOG}.{SCHEMA}.physical_resource_span").filter(F.col("is_active") == True)
spans = spans_df.select("span_id", "site_a_id", "site_z_id", "length_km", "conduit_id").collect()

sites_df = spark.table(f"{CATALOG}.{SCHEMA}.geographic_site")
site_coords = {row.site_id: (float(row.latitude), float(row.longitude))
               for row in sites_df.select("site_id", "latitude", "longitude").collect()}

# Build adjacency graph
adj = defaultdict(set)
edge_spans = defaultdict(list)

for row in spans:
    a, z = row.site_a_id, row.site_z_id
    adj[a].add(z)
    adj[z].add(a)
    edge_key = tuple(sorted([a, z]))
    edge_spans[edge_key].append({
        "span_id": row.span_id, "site_a": a, "site_z": z,
        "length_km": float(row.length_km) if row.length_km else 0,
        "conduit_id": row.conduit_id
    })

sites = list(adj.keys())
print(f"BEFORE augmentation: {len(sites)} sites, {len(edge_spans)} edges, {len(spans)} spans")

# --- Find connected components ---
def find_components(graph, nodes):
    visited = set()
    comps = []
    for node in nodes:
        if node not in visited:
            comp = set()
            q = deque([node])
            while q:
                n = q.popleft()
                if n not in comp:
                    comp.add(n)
                    visited.add(n)
                    for nb in graph.get(n, set()):
                        if nb not in comp:
                            q.append(nb)
            comps.append(comp)
    comps.sort(key=len, reverse=True)
    return comps

comps = find_components(adj, sites)
print(f"Connected components: {len(comps)} (sizes: {[len(c) for c in comps]})")
print(f"Graph is {'DISCONNECTED' if len(comps) > 1 else 'connected'} — augmentation needed")

In [0]:
# --- Haversine distance for realistic span lengths ---
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

# --- Generate backbone spans to connect all components ---
# Strategy: chain component representatives, then add ring links for diversity
def pick_representative(comp):
    """Pick a site with highest degree as component representative."""
    return max(comp, key=lambda s: len(adj.get(s, set())))

comp_reps = [pick_representative(c) for c in comps]
print(f"Component representatives: {comp_reps}")

# Conduit data for new spans
conduits = list(spark.table(f"{CATALOG}.{SCHEMA}.physical_resource_conduit").select("conduit_id").toPandas()["conduit_id"])

new_span_rows = []
span_counter = 10001
now_ts = datetime.utcnow()

def make_backbone_spans(site_a, site_z, count=4):
    """Create multiple parallel spans between two sites for realistic capacity."""
    global span_counter
    rows = []
    dist = haversine_km(*site_coords.get(site_a, (39,-98)), *site_coords.get(site_z, (39,-98)))
    for i in range(count):
        span_id = f"BB-{span_counter:06d}"
        span_counter += 1
        conduit_id = random.choice(conduits) if i < count // 2 else random.choice(conduits)  # Mix conduits
        fiber_type = random.choice(["SMF-28e", "LEAF", "TrueWave"])
        # Generate WKT LINESTRING from site coords
        c_a = site_coords.get(site_a, (39.0, -98.0))
        c_z = site_coords.get(site_z, (39.0, -95.0))
        wkt = f"LINESTRING ({c_a[1]} {c_a[0]}, {c_z[1]} {c_z[0]})"
        rows.append((
            span_id, f"Backbone {site_a}-{site_z} #{i+1}", conduit_id,
            site_a, site_z, fiber_type,
            random.randint(48, 432), random.randint(8, 80),
            round(dist * random.uniform(1.1, 1.3), 1),  # Route factor
            wkt, random.randint(2015, 2024), True, now_ts
        ))
    return rows

# 1) Chain all components: rep[0]->rep[1]->...->rep[N-1]
for i in range(len(comp_reps) - 1):
    spans_new = make_backbone_spans(comp_reps[i], comp_reps[i+1], count=4)
    new_span_rows.extend(spans_new)
    # Update adjacency
    adj[comp_reps[i]].add(comp_reps[i+1])
    adj[comp_reps[i+1]].add(comp_reps[i])
    ek = tuple(sorted([comp_reps[i], comp_reps[i+1]]))
    edge_spans[ek].extend([{
        "span_id": r[0], "site_a": r[3], "site_z": r[4],
        "length_km": r[8], "conduit_id": r[2]
    } for r in spans_new])

# 2) Add ring-closure links for diversity (connect every 3rd component)
for i in range(0, len(comp_reps) - 2, 3):
    j = min(i + 3, len(comp_reps) - 1)
    if j != i + 1:  # Avoid duplicate of chain edge
        spans_new = make_backbone_spans(comp_reps[i], comp_reps[j], count=3)
        new_span_rows.extend(spans_new)
        adj[comp_reps[i]].add(comp_reps[j])
        adj[comp_reps[j]].add(comp_reps[i])
        ek = tuple(sorted([comp_reps[i], comp_reps[j]]))
        edge_spans[ek].extend([{
            "span_id": r[0], "site_a": r[3], "site_z": r[4],
            "length_km": r[8], "conduit_id": r[2]
        } for r in spans_new])

# 3) Add intra-component ring links (connect leaf nodes to other nodes)
for comp in comps:
    leaf_nodes = [s for s in comp if len(adj[s]) == 1]
    non_leaf = [s for s in comp if len(adj[s]) > 1]
    if len(leaf_nodes) >= 2 and non_leaf:
        # Connect first leaf to a non-adjacent non-leaf
        for leaf in leaf_nodes[:2]:
            candidates = [n for n in non_leaf if n not in adj[leaf]]
            if candidates:
                target = min(candidates, key=lambda t: haversine_km(
                    *site_coords.get(leaf, (39,-98)), *site_coords.get(t, (39,-98))))
                spans_new = make_backbone_spans(leaf, target, count=2)
                new_span_rows.extend(spans_new)
                adj[leaf].add(target)
                adj[target].add(leaf)
                ek = tuple(sorted([leaf, target]))
                edge_spans[ek].extend([{
                    "span_id": r[0], "site_a": r[3], "site_z": r[4],
                    "length_km": r[8], "conduit_id": r[2]
                } for r in spans_new])

print(f"\nGenerated {len(new_span_rows)} backbone/ring spans")

# Verify connectivity
comps_after = find_components(adj, list(adj.keys()))
print(f"AFTER augmentation: {len(adj)} sites, {len(edge_spans)} edges")
print(f"Connected components: {len(comps_after)} (sizes: {[len(c) for c in comps_after]})")
print(f"Graph is now {'CONNECTED' if len(comps_after) == 1 else 'STILL DISCONNECTED'}")

In [0]:
# --- BFS and path algorithms ---
def bfs_shortest_path(graph, start, end, excluded_edges=None):
    """BFS shortest path returning list of sites."""
    if start == end:
        return [start]
    excluded = excluded_edges or set()
    visited = {start}
    queue = deque([(start, [start])])
    while queue:
        node, path = queue.popleft()
        for neighbor in graph.get(node, set()):
            edge_key = tuple(sorted([node, neighbor]))
            if neighbor not in visited and edge_key not in excluded:
                new_path = path + [neighbor]
                if neighbor == end:
                    return new_path
                visited.add(neighbor)
                queue.append((neighbor, new_path))
    return None

def get_edge_disjoint_path(graph, start, end, primary_path):
    """Find alternate path avoiding primary path edges."""
    if not primary_path or len(primary_path) < 2:
        return None
    excluded = {tuple(sorted([primary_path[i], primary_path[i+1]])) for i in range(len(primary_path)-1)}
    alt = bfs_shortest_path(graph, start, end, excluded_edges=excluded)
    if alt is None:
        # Fallback: reverse BFS (may share some edges but different span selection)
        alt = bfs_shortest_path(graph, end, start)
    return alt

def select_span_for_edge(site_from, site_to, edge_spans_map, used_conduits=None):
    """Select a span for an edge, preferring conduit diversity."""
    edge_key = tuple(sorted([site_from, site_to]))
    candidates = edge_spans_map.get(edge_key, [])
    if not candidates:
        return None
    if used_conduits:
        diverse = [s for s in candidates if s["conduit_id"] not in used_conduits]
        if diverse:
            candidates = diverse
    return random.choice(candidates)

# --- Test path finding on connected graph ---
all_sites = list(adj.keys())
test_path = bfs_shortest_path(adj, all_sites[0], all_sites[-1])
print(f"Test BFS: {all_sites[0]} -> {all_sites[-1]}: {'SUCCESS' if test_path else 'FAILED'} ({len(test_path) if test_path else 0} hops)")

# --- Generate paths for all 8K services ---
services_df = spark.table(f"{CATALOG}.{SCHEMA}.service_inventory")
svc_rows = services_df.select("service_id", "sold_as_diverse", "site_a_id", "site_z_id").collect()
services = [row.service_id for row in svc_rows]
is_diverse = {row.service_id: row.sold_as_diverse for row in svc_rows}
svc_endpoints = {row.service_id: (row.site_a_id, row.site_z_id) for row in svc_rows}

path_rows = []
stats = {"working": 0, "protect": 0, "no_path": 0, "no_alt": 0, "fallback_protect": 0}

for svc_id in services:
    # Use service's actual site endpoints; fallback to hash if sites not in graph
    ep = svc_endpoints.get(svc_id, (None, None))
    origin = ep[0] if ep[0] in adj else all_sites[hash(svc_id) % len(all_sites)]
    dest = ep[1] if ep[1] in adj else all_sites[(hash(svc_id) // len(all_sites)) % len(all_sites)]
    if origin == dest:
        dest = all_sites[(all_sites.index(origin) + 1) % len(all_sites)]
    
    working_path = bfs_shortest_path(adj, origin, dest)
    if not working_path or len(working_path) < 2:
        stats["no_path"] += 1
        continue
    
    w_path_id = f"WP-{svc_id[-8:]}"
    used_conduits = set()
    for i in range(len(working_path) - 1):
        span_rec = select_span_for_edge(working_path[i], working_path[i+1], edge_spans, used_conduits)
        if span_rec:
            used_conduits.add(span_rec["conduit_id"])
            path_rows.append((w_path_id, svc_id, "working", span_rec["span_id"], i+1, now_ts))
    stats["working"] += 1
    
    if is_diverse.get(svc_id, False):
        protect_path = get_edge_disjoint_path(adj, origin, dest, working_path)
        if protect_path and len(protect_path) >= 2:
            p_path_id = f"PP-{svc_id[-8:]}"
            protect_conduits = set()
            for i in range(len(protect_path) - 1):
                span_rec = select_span_for_edge(
                    protect_path[i], protect_path[i+1], edge_spans,
                    used_conduits=used_conduits | protect_conduits
                )
                if span_rec:
                    protect_conduits.add(span_rec["conduit_id"])
                    path_rows.append((p_path_id, svc_id, "protect", span_rec["span_id"], i+1, now_ts))
            # Track if protect used fallback (same edges, different spans)
            w_edges = {tuple(sorted([working_path[i], working_path[i+1]])) for i in range(len(working_path)-1)}
            p_edges = {tuple(sorted([protect_path[i], protect_path[i+1]])) for i in range(len(protect_path)-1)}
            if w_edges == p_edges:
                stats["fallback_protect"] += 1
            stats["protect"] += 1
        else:
            stats["no_alt"] += 1

print(f"\nPath generation complete:")
print(f"  Working paths: {stats['working']}")
print(f"  Protect paths: {stats['protect']} ({stats['fallback_protect']} fallback/same-route)")
print(f"  No path found: {stats['no_path']}")
print(f"  No alt found: {stats['no_alt']}")
print(f"  Total hop rows: {len(path_rows)}")
print(f"  Services: {len(services)}, diverse: {sum(1 for v in is_diverse.values() if v)}")

In [0]:
# --- Step 1: Write backbone spans to physical_resource_span ---
if new_span_rows:
    span_schema = StructType([
        StructField("span_id", StringType(), False),
        StructField("span_name", StringType(), True),
        StructField("conduit_id", StringType(), True),
        StructField("site_a_id", StringType(), True),
        StructField("site_z_id", StringType(), True),
        StructField("fiber_type", StringType(), True),
        StructField("fiber_count", IntegerType(), True),
        StructField("wavelength_count", IntegerType(), True),
        StructField("length_km", DoubleType(), True),
        StructField("geom_wkt", StringType(), True),
        StructField("install_year", IntegerType(), True),
        StructField("is_active", BooleanType(), True),
        StructField("created_at", TimestampType(), True)
    ])
    new_spans_df = spark.createDataFrame(new_span_rows, span_schema)
    new_spans_df.write.mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.physical_resource_span")
    total_spans = spark.table(f"{CATALOG}.{SCHEMA}.physical_resource_span").count()
    print(f"Appended {len(new_span_rows)} backbone spans. Total: {total_spans}")

# --- Step 2: Build service path DataFrame ---
path_schema = StructType([
    StructField("path_id", StringType(), False),
    StructField("service_id", StringType(), False),
    StructField("path_type", StringType(), False),
    StructField("span_id", StringType(), False),
    StructField("hop_sequence", IntegerType(), False),
    StructField("created_at", TimestampType(), False)
])
new_paths_df = spark.createDataFrame(path_rows, path_schema)

# --- Step 3: Validate topological continuity ---
# Build combined span site lookup (original + backbone)
all_spans = spark.table(f"{CATALOG}.{SCHEMA}.physical_resource_span")
span_sites = {row.span_id: (row.site_a_id, row.site_z_id)
              for row in all_spans.select("span_id", "site_a_id", "site_z_id").collect()}

path_hops_map = defaultdict(list)
for row in path_rows:
    path_hops_map[row[0]].append((row[4], row[3]))

continuous = total_pairs = 0
for path_id, hops in path_hops_map.items():
    hops.sort(key=lambda x: x[0])
    for i in range(len(hops) - 1):
        s1 = span_sites.get(hops[i][1])
        s2 = span_sites.get(hops[i+1][1])
        if s1 and s2:
            if set(s1) & set(s2):
                continuous += 1
            total_pairs += 1

continuity_pct = 100 * continuous / total_pairs if total_pairs else 0
print(f"\nTopological continuity: {continuous}/{total_pairs} ({continuity_pct:.1f}%)")
print(f"Target: 100%. {'PASS' if continuity_pct == 100 else 'NEEDS INVESTIGATION'}")

print(f"\nNew paths: {len(path_rows)} rows")
new_paths_df.groupBy("path_type").agg(
    F.countDistinct("path_id").alias("paths"),
    F.count("*").alias("hops"),
    F.avg("hop_sequence").alias("avg_hop"),
    F.max("hop_sequence").alias("max_hop")
).show()

In [0]:
# INSERT OVERWRITE to replace random paths with topologically valid ones
new_paths_df.createOrReplaceTempView("new_service_paths")

spark.sql(f"""
    INSERT OVERWRITE {CATALOG}.{SCHEMA}.service_path
    SELECT path_id, service_id, path_type, span_id, hop_sequence, created_at
    FROM new_service_paths
""")

final_count = spark.table(f"{CATALOG}.{SCHEMA}.service_path").count()
print(f"service_path: {final_count} rows (was 88,047)")

# --- Full DQ health check ---
print("\n=== DQ HEALTH CHECK ===")

# 1. FK integrity: service_id
result = spark.sql(f"""
    SELECT COUNT(*) as orphan_service_ids
    FROM {CATALOG}.{SCHEMA}.service_path sp
    LEFT JOIN {CATALOG}.{SCHEMA}.service_inventory si ON sp.service_id = si.service_id
    WHERE si.service_id IS NULL
""")
orphan_svc = result.collect()[0].orphan_service_ids
print(f"FK service_id orphans: {orphan_svc} {'PASS' if orphan_svc == 0 else 'FAIL'}")

# 2. FK integrity: span_id  
result = spark.sql(f"""
    SELECT COUNT(*) as orphan_span_ids
    FROM {CATALOG}.{SCHEMA}.service_path sp
    LEFT JOIN {CATALOG}.{SCHEMA}.physical_resource_span s ON sp.span_id = s.span_id
    WHERE s.span_id IS NULL
""")
orphan_span = result.collect()[0].orphan_span_ids
print(f"FK span_id orphans: {orphan_span} {'PASS' if orphan_span == 0 else 'FAIL'}")

# 3. Topological continuity (full check)
result = spark.sql(f"""
    WITH path_hops AS (
      SELECT sp.path_id, sp.hop_sequence, s.site_a_id, s.site_z_id
      FROM {CATALOG}.{SCHEMA}.service_path sp
      JOIN {CATALOG}.{SCHEMA}.physical_resource_span s ON sp.span_id = s.span_id
    ),
    consecutive AS (
      SELECT 
        CASE WHEN a.site_z_id = b.site_a_id OR a.site_z_id = b.site_z_id 
             OR a.site_a_id = b.site_a_id OR a.site_a_id = b.site_z_id 
             THEN 1 ELSE 0 END as shared_site
      FROM path_hops a
      JOIN path_hops b ON a.path_id = b.path_id AND b.hop_sequence = a.hop_sequence + 1
    )
    SELECT COUNT(*) as total_pairs, SUM(shared_site) as continuous,
           ROUND(100.0 * SUM(shared_site) / COUNT(*), 1) as pct
    FROM consecutive
""")
cont = result.collect()[0]
print(f"Topological continuity: {cont.continuous}/{cont.total_pairs} ({cont.pct}%) {'PASS' if cont.pct == 100.0 else 'INVESTIGATE'}")

# 4. Service coverage
result = spark.sql(f"""
    SELECT 
      (SELECT COUNT(DISTINCT service_id) FROM {CATALOG}.{SCHEMA}.service_path) as path_services,
      (SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.service_inventory) as total_services
""")
cov = result.collect()[0]
print(f"Service coverage: {cov.path_services}/{cov.total_services} ({100*cov.path_services/cov.total_services:.1f}%)")

# 5. Path type distribution
spark.sql(f"""
    SELECT path_type, COUNT(DISTINCT path_id) as paths, COUNT(DISTINCT service_id) as services,
           COUNT(*) as hops, ROUND(AVG(hop_sequence),1) as avg_hop, MAX(hop_sequence) as max_hop
    FROM {CATALOG}.{SCHEMA}.service_path GROUP BY path_type
""").show()

print("\n=== SERVICE PATH DECOMPOSITION COMPLETE ===")